**N3a_fitting_withRR**: This notebook creates the simulation object, which gathers all the information, and fits the alphas to the observation given with a NNLS with ridge-regularization.

### 1. Initialization

Imports and general parameters (from *parameters.py*).

In [ ]:
## ----- IPYTHON COMMANDS ----- ##
%load_ext autoreload
%autoreload 2


In [ ]:
## ----- IMPORTS ----- ##
import satellite_RFI.src.simulation as sim
from scipy.optimize import nnls
import sys
sys.path.insert(0, './initialization/')
from imports import *
import parameters as pm
from pathlib import Path


In [ ]:
## ----- FITTING PARAMETERS ----- ##

# folder in which to save results
folder = ""
Path(folder).mkdir(exist_ok=True)  # <-- if it doesn't exist, create it

# simulation information
path_beam = "{}satbeams_{}_{}_{}-{}.pkl".format(pm.folder, pm.block, pm.beam_model, *pm.freq_range)
path_catalog = f"{pm.folder}catalog_{pm.block}.csv"
path_nearby = f"{pm.folder}nearby_{pm.block}.pkl"

# correct label for satellites in the catalog
if path_catalog==f"{pm.folder}catalogOLD_{pm.block}.csv":  label_sats = "Sat"
else:  label_sats = "NORAD ID"

# cost function from eq. 11 (options: "C1"=radiometer, "C2"=1)
CF = "C1"
RR = 

# angular mask [degrees] (options: 1,5 or None)
deg = None
if deg!=None:  times_nearby  = pickle.load(open(path_nearby, "rb"))[deg]
else:  times_nearby = None

# thermal mask [kelvin] (options: 25,50,100 or None)
temp = None

# threshold pixel mask (options: 2,5,7 or None)
pix = None

# temporal mask [seconds] (options: 775-1000, 2200-2400, 5500-6200 or None)
time_slice = (None,None)


In [ ]:
pm.show_parameters(CF, deg, temp, pix, time_slice)
print()
fname = pm.my_name(folder, CF, deg, temp, pix, time_slice)
print("File with final alpha parameters will be '{}'".format(fname))


### 2. Setting up the simulation

We initialize the SatelliteSimulation object, which will store the information. 

In [ ]:
# initializing the simulation
sat = sim.SatelliteSimulation(
    survey_info=[pm.nd_s0, pm.frequency],
    path_catalog=path_catalog,
    path_beam=path_beam,
    freq_range=pm.freq_range,
    freq_slice=pm.freq_slice,
    time_slice=time_slice,
    label_sats=label_sats,
    verbose=True,
)

# getting observations
sat.use_observations(path_observations=pm.path_observations, verbose=True)

# applying mask
print("Masking...")
sat.use_mask(times_nearby, temp, pix, verbose=True)


In [ ]:
# defining the cost function (only for testing, this won't be used in the optimization)
if CF=="C1":
    def Cost_Function(alphas):
        ''' Computes CF for the given alphas, with weights=obs (case C1). '''
        sat.simulate(alphas)
        CF = np.sum( ((sat.obs_BGsub-sat.sim) / sat.obs)**2 )
        print(CF,end="\t")
        return CF
elif CF=="C2":
    def Cost_Function(alphas):
        ''' Computes CF for the given alphas, with weights=1 (case C2). '''
        sat.simulate(alphas)
        CF = np.sum( (sat.obs_BGsub-sat.sim)**2 )
        print(CF,end="\t")
        return CF

# defining initial values (only for testing, this won't be used in the optimization)
alphas0 = np.zeros(len(sat.catalog))
_ = Cost_Function(alphas0)


### 3. Running the optimization

#### A. Getting values

We now run the optimization with the functions and parameters defined above. This part of the code **needs approximately 15GB** of memory to run the `nnls` optimization, and it takes some minutes to run the code for the creation of matrixes A and b.

In [ ]:
def optimization_setup(CF, RR, verbose=True):  
    ''' Computes the necessary matrices A,b,Anorm for the nnls minimization;
    if lamb!=0, performs ridge regularization. '''

    # in each satellite, loop through the specific signals
    A = np.empty((sat.obs_BGsub.size, len(sat.catalog)))
    if verbose:  print("Creating matrix A and b ...")
    for i_sat, start in enumerate(sat.index_sats):
        stop = start + sat.n_signals[i_sat]
        for i in range(start, stop):
            A[:, i] = (sat.sat_beam[i_sat] * sat.Tb_factors[i][:,None]).ravel()
    b = sat.obs_BGsub.ravel()

    # checking dimensions
    if verbose: 
        print(f" - Size of matrix A: {A.nbytes / 1024**3:.3f} GB")
        print(f" - Size of vector b: {b.nbytes / 1024**3:.3f} GB")

    # defining weights if necessary
    if verbose:  print("Applying weights...")
    if CF=="C1":  
        weights = 1.0 / sat.obs.ravel()
        A *= weights[:,None]
        b *= weights

    # applying ridge-regularization
    if verbose:  print(f"Applying ridge regularization (λ = {RR:g})")
    A = np.vstack([A, np.sqrt(RR) * np.eye(A.shape[1])])
    b = np.concatenate([b, np.sqrt(RR) * np.ones(A.shape[1])])
    
    return A,b


In [ ]:
# setting up the optimization (creating matrixes A,b)
A,b = optimization_setup(CF,RR)


In [ ]:
# running the optimization
start = time.perf_counter()
print("Running optimization...")
x,rnorm = nnls(A, b, maxiter=1000)
print("Done.")
elapsed = time.perf_counter() - start
print(f"This took {elapsed:.2f} seconds, or {elapsed/60:.2f} minutes")


#### B. Plotting best-fit

In [ ]:
# saving best fit values
alphas_BF = x

# simulating
print("Simulating...")
sat = sim.SatelliteSimulation(
    survey_info=[pm.nd_s0, pm.frequency],
    path_catalog=path_catalog,
    path_beam=path_beam,
    freq_range=pm.freq_range,
    freq_slice=pm.freq_slice,
    time_slice=time_slice,
    label_sats=label_sats,
    verbose=False,
)

# getting observations
sat.use_observations(path_observations=pm.path_observations, verbose=False)

# applying mask
sat.use_mask(times_nearby, temp, pix, verbose=False)

# using alphas
sat.simulate(alphas_BF)


In [ ]:
# plotting values
plt.plot(alphas_BF,"k.")
plt.title("Best fit obtained")
plt.show()

# plotting simulation
f = sat.frequency[sat.ifreq[0]:sat.ifreq[1]]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(f, np.ma.mean(sat.obs_BGsub.T + sat.BG.T, axis=0), label='Observation')
ax.plot(f, np.ma.mean(sat.sim.T + sat.BG.T, axis=0), '--', label='Simulation')
func = Cost_Function(alphas_BF)/sat.obs_BGsub.size + np.sum(RR*(1-alphas_BF)**2)
textstr = f"$\\sigma_D={CF}$\nFoM={func:.3f}"
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
ax.text(0.015, 0.95, textstr, transform=ax.transAxes, fontsize=18, verticalalignment='top', bbox=props)
ax.set_xlabel('Frequency [MHz]')
ax.set_ylabel('Temperature [K]')
ax.legend(loc='upper right')
plt.show()

# plotting simulation
t = sat.time[sat.itime[0]:sat.itime[1]]
fig, ax = plt.subplots(figsize=(8, 6))
res = sat.obs_BGsub.T - sat.sim.T
lim = np.nanpercentile(np.abs(res), 99.5)
print(f"Maximum found was {lim}.")
cax = ax.imshow(res, aspect='auto', extent=[f[0],f[-1],t[-1],t[0]], 
                cmap="RdBu_r", vmin=-lim, vmax=lim)
cbar = fig.colorbar(cax, ax=ax)
cbar.set_label(r'Temperature [K]', rotation=270, labelpad=20)
ax.set_ylabel('Time [sec]')
ax.set_xlabel('Frequency [MHz]')
plt.title("Residuals")
plt.show()


In [ ]:
# saving information in the file
data_info = {
    "initial" : alphas0,
    "CF" : CF,
    "mask_degree" : deg, 
    "mask_temperature" : temp, 
    "mask_pix" : pix, 
    "time_slice" : time_slice,
    "frequency_slice" : pm.freq_slice, 
    "best-fit" : alphas_BF,
}

print(f"Information saved in file '{fname}'.")
pickle.dump( data_info, open(fname,"wb") )
